In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- repo paths ---
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent
SCRIPTS_DIR = REPO_ROOT / "scripts"
SRC_DIR = REPO_ROOT / "src"

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from bandit_instance import build_bandit_instance_from_run, summarize_instance
from bandit_eval import (
    oracle_action_indices,
    random_action_indices,
    greedy_action_indices_from_reward_source,
    evaluate_action_indices,
    cumulative_sum,
)

# ============================================================
# Choose your saved embedding run here
# ============================================================
RUN_DIR = REPO_ROOT / "results" / "final_embedding_divisor_hidden_512"
# ^ if this folder actually contains your chosen 512 divisor run, that's fine.
# just point to the real folder you want to use.

# reward_mode:
#   "oracle"     -> misspecified real-data bandit instance
#   "linearized" -> exact linearized bandit instance from ridge fit
REWARD_MODE = "oracle"

RANDOM_BASELINE_TRIALS = 50
RANDOM_BASELINE_SEED0 = 123

print("RUN_DIR:", RUN_DIR)
print("REWARD_MODE:", REWARD_MODE)

RUN_DIR: d:\2023-2028_UCLA_Research_Projects\Bandits\21-1-26-ICML Attempt\real-data-tests\results\final_embedding_divisor_hidden_512
REWARD_MODE: oracle


In [2]:
instance = build_bandit_instance_from_run(
    RUN_DIR,
    reward_mode=REWARD_MODE,
    include_intercept=True,
    sort_rows=True,
)

summary = summarize_instance(instance)
summary_df = pd.DataFrame([summary]).T
display(summary_df)

print("\nBasic checks")
print("num_times:", instance.num_times)
print("feature_dim:", instance.feature_dim)
print("actions at t=0:", instance.X_by_t[0].shape[0])
print("X[0] shape:", instance.X_by_t[0].shape)
print("theta_star shape:", instance.theta_star.shape)
print("first 5 subset labels at t=0:", instance.subset_strs_by_t[0][:5])

,0
num_times,7473
feature_dim,25
num_actions_min,129
num_actions_max,129
num_actions_mean,129.0
reward_mode,oracle
oracle_reward_mean,0.489381
oracle_reward_std,0.180569
linearized_reward_mean,0.488149
linearized_reward_std,0.178997



Basic checks
num_times: 7473
feature_dim: 25
actions at t=0: 129
X[0] shape: (129, 25)
theta_star shape: (25,)
first 5 subset labels at t=0: ['1', '10', '2', '3', '4']


In [ ]:
# ============================================================
# Deterministic baselines
# ============================================================
oracle_actions = oracle_action_indices(instance, reward_source=REWARD_MODE)
oracle_eval = evaluate_action_indices(
    instance,
    oracle_actions,
    evaluation_source=REWARD_MODE,
    realized_source=REWARD_MODE,
    ks=(1, 3),
)

linearized_greedy_actions = greedy_action_indices_from_reward_source(instance, reward_source="linearized")
linearized_greedy_eval = evaluate_action_indices(
    instance,
    linearized_greedy_actions,
    evaluation_source=REWARD_MODE,
    realized_source=REWARD_MODE,
    ks=(1, 3),
)

# ============================================================
# Random baseline averaged over several trials
# ============================================================
random_summaries = []
random_regret_traces = []

for trial in range(RANDOM_BASELINE_TRIALS):
    rand_actions = random_action_indices(instance, seed=RANDOM_BASELINE_SEED0 + trial)
    rand_eval = evaluate_action_indices(
        instance,
        rand_actions,
        evaluation_source=REWARD_MODE,
        realized_source=REWARD_MODE,
        ks=(1, 3),
    )
    random_summaries.append(rand_eval["summary"])
    random_regret_traces.append(rand_eval["regrets"])

random_df = pd.DataFrame(random_summaries)
random_summary = random_df.mean(numeric_only=True).to_dict()
random_regret_mean_trace = np.mean(np.stack(random_regret_traces), axis=0)

# ============================================================
# Collect comparison table
# ============================================================
compare = pd.DataFrame([
    {"method": "oracle", **oracle_eval["summary"]},
    {"method": "linearized_greedy", **linearized_greedy_eval["summary"]},
    {"method": f"random_uniform_mean_{RANDOM_BASELINE_TRIALS}trials", **random_summary},
])

display(compare)

# ============================================================
# Plot cumulative regret traces
# ============================================================
oracle_cum_regret = cumulative_sum(oracle_eval["regrets"])
lin_cum_regret = cumulative_sum(linearized_greedy_eval["regrets"])
rand_cum_regret = cumulative_sum(random_regret_mean_trace)

plt.figure(figsize=(10, 5))
plt.plot(instance.times, oracle_cum_regret, label="oracle")
plt.plot(instance.times, lin_cum_regret, label="linearized_greedy")
plt.plot(instance.times, rand_cum_regret, label=f"random_mean_{RANDOM_BASELINE_TRIALS}")
plt.title(f"Cumulative regret ({REWARD_MODE} rewards)")
plt.xlabel("time index")
plt.ylabel("cumulative regret")
plt.legend()
plt.tight_layout()
plt.show()